In [8]:
import os
import joblib
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import fisher_exact

# === Background genes ===
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.str.upper())

# === Cell types ===
cell_types = ['Ast', 'Mic', 'Inh', 'Oli', 'Ex', 'Opc']

# === Paths (DEMOGRAPHICS versions) ===
ad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_new"
cerad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_seaad"

# === Loop ===
for cell_type in cell_types:
    print(f"\n{'='*60}")
    print(f"Cell type: {cell_type}")
    print(f"{'='*60}")

    ad_cell_type = "In" if cell_type == "Inh" else cell_type

    # --- AD predictors ---
    gene_presence_ad = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(ad_base_dir, ad_cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_presence_ad[gene.upper()] += 1
    ad_predictors = {g for g, c in gene_presence_ad.items() if c >= 2}

    # --- Validation predictors ---
    gene_presence_cerad = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(cerad_base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_presence_cerad[gene.upper()] += 1
    cerad_predictors = {g for g, c in gene_presence_cerad.items() if c >= 2}

    # --- Fisher test ---
    overlap = ad_predictors & cerad_predictors

    a = len(overlap)
    b = len(ad_predictors - overlap)
    c = len(cerad_predictors - overlap)
    d = len(background_genes - (ad_predictors | cerad_predictors))

    table = [[a, b], [c, d]]
    odds_ratio, p_value = fisher_exact(table, alternative='greater')

    print(f"AD predictors: {len(ad_predictors)}")
    print(f"Validation predictors: {len(cerad_predictors)}")
    print(f"Overlap: {a}")
    print(f"Odds Ratio: {odds_ratio:.4f}")
    print(f"P-value: {p_value:.4e}")
    print(f"Overlap genes ({len(overlap)}):")
    print(sorted(overlap))


Cell type: Ast
AD predictors: 77
Validation predictors: 21
Overlap: 2
Odds Ratio: 25.0456
P-value: 3.6215e-03
Overlap genes (2):
['ARL17B', 'FTH1']

Cell type: Mic
AD predictors: 202
Validation predictors: 178
Overlap: 28
Odds Ratio: 18.9113
P-value: 3.2503e-24
Overlap genes (28):
['ARHGAP24', 'ARHGAP26', 'ARL17B', 'CARD11', 'CELF2', 'CPVL', 'DPYD', 'FCHSD2', 'FOXP1', 'HCLS1', 'JAZF1', 'KANSL1', 'KCNMA1', 'MAP3K8', 'MEF2A', 'NFAT5', 'PICALM', 'PIK3R5', 'PLXDC2', 'RHBDF2', 'RNF150', 'RPS19', 'SLC11A1', 'SORL1', 'SPP1', 'ST6GAL1', 'TBC1D5', 'ZC3HAV1']

Cell type: Inh
AD predictors: 22
Validation predictors: 16
Overlap: 1
Odds Ratio: 56.8508
P-value: 1.9444e-02
Overlap genes (1):
['USP9Y']

Cell type: Oli
AD predictors: 78
Validation predictors: 10
Overlap: 1
Odds Ratio: 25.7576
P-value: 4.2655e-02
Overlap genes (1):
['QDPR']

Cell type: Ex
AD predictors: 10
Validation predictors: 5
Overlap: 0
Odds Ratio: 0.0000
P-value: 1.0000e+00
Overlap genes (0):
[]

Cell type: Opc
AD predictors: 74


In [ ]:
============================================================
Cell type: Ast
============================================================
AD predictors: 77
Validation predictors: 21
Overlap: 2
Odds Ratio: 25.0456
P-value: 3.6215e-03
Overlap genes (2):
['ARL17B', 'FTH1']

============================================================
Cell type: Mic
============================================================
AD predictors: 202
Validation predictors: 178
Overlap: 28
Odds Ratio: 18.9113
P-value: 3.2503e-24
Overlap genes (28):
['ARHGAP24', 'ARHGAP26', 'ARL17B', 'CARD11', 'CELF2', 'CPVL', 'DPYD', 'FCHSD2', 'FOXP1', 'HCLS1', 'JAZF1', 'KANSL1', 'KCNMA1', 'MAP3K8', 'MEF2A', 'NFAT5', 'PICALM', 'PIK3R5', 'PLXDC2', 'RHBDF2', 'RNF150', 'RPS19', 'SLC11A1', 'SORL1', 'SPP1', 'ST6GAL1', 'TBC1D5', 'ZC3HAV1']

============================================================
Cell type: Inh
============================================================
AD predictors: 22
Validation predictors: 16
Overlap: 1
Odds Ratio: 56.8508
P-value: 1.9444e-02
Overlap genes (1):
['USP9Y']

============================================================
Cell type: Oli
============================================================
AD predictors: 78
Validation predictors: 10
Overlap: 1
Odds Ratio: 25.7576
P-value: 4.2655e-02
Overlap genes (1):
['QDPR']

============================================================
Cell type: Ex
============================================================
AD predictors: 10
Validation predictors: 5
Overlap: 0
Odds Ratio: 0.0000
P-value: 1.0000e+00
Overlap genes (0):
[]

============================================================
Cell type: Opc
============================================================
AD predictors: 74
Validation predictors: 13
Overlap: 1
Odds Ratio: 20.3836
P-value: 5.2327e-02
Overlap genes (1):
['UTY']

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import fisher_exact

# === Background genes ===
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.str.upper())

# === Cell types ===
cell_types = ['Ast', 'Mic', 'Inh', 'Oli', 'Ex', 'Opc']

# === Paths (DEMOGRAPHICS versions) ===
ad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
cerad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad/"

# === Loop ===
for cell_type in cell_types:
    print(f"\n{'='*60}")
    print(f"Cell type: {cell_type}")
    print(f"{'='*60}")

    

    ad_cell_type = "In" if cell_type == "Inh" else cell_type

    # --- AD predictors ---
    gene_presence_ad = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(ad_base_dir, ad_cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_presence_ad[gene.upper()] += 1
    ad_predictors = {g for g, c in gene_presence_ad.items() if c >= 2}

    # --- Validation predictors ---
    gene_presence_cerad = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(cerad_base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_presence_cerad[gene.upper()] += 1
    cerad_predictors = {g for g, c in gene_presence_cerad.items() if c >= 2}

    # --- Fisher test ---
    overlap = ad_predictors & cerad_predictors

    a = len(overlap)
    b = len(ad_predictors - overlap)
    c = len(cerad_predictors - overlap)
    d = len(background_genes - (ad_predictors | cerad_predictors))

    table = [[a, b], [c, d]]
    odds_ratio, p_value = fisher_exact(table, alternative='greater')

    print(f"AD predictors: {len(ad_predictors)}")
    print(f"Validation predictors: {len(cerad_predictors)}")
    print(f"Overlap: {a}")
    print(f"Odds Ratio: {odds_ratio:.4f}")
    print(f"P-value: {p_value:.4e}")
    print(f"Overlap genes ({len(overlap)}):")
    print(sorted(overlap))


Cell type: Ast
AD predictors: 175
Validation predictors: 0
Overlap: 0
Odds Ratio: nan
P-value: 1.0000e+00
Overlap genes (0):
[]

Cell type: Mic
AD predictors: 466
Validation predictors: 0
Overlap: 0
Odds Ratio: nan
P-value: 1.0000e+00
Overlap genes (0):
[]

Cell type: Inh
AD predictors: 108
Validation predictors: 0
Overlap: 0
Odds Ratio: nan
P-value: 1.0000e+00
Overlap genes (0):
[]

Cell type: Oli
AD predictors: 621
Validation predictors: 0
Overlap: 0
Odds Ratio: nan
P-value: 1.0000e+00
Overlap genes (0):
[]

Cell type: Ex
AD predictors: 162
Validation predictors: 0
Overlap: 0
Odds Ratio: nan
P-value: 1.0000e+00
Overlap genes (0):
[]

Cell type: Opc
AD predictors: 835
Validation predictors: 0
Overlap: 0
Odds Ratio: nan
P-value: 1.0000e+00
Overlap genes (0):
[]


In [ ]:
============================================================
Cell type: Ast
============================================================
AD predictors: 175
Validation predictors: 18
Overlap: 3
Odds Ratio: 20.6337
P-value: 6.6897e-04
Overlap genes (3):
['FTH1', 'MRPS6', 'UTY']

============================================================
Cell type: Mic
============================================================
AD predictors: 466
Validation predictors: 61
Overlap: 17
Odds Ratio: 15.0062
P-value: 1.5548e-13
Overlap genes (17):
['ADGRB3', 'APOE', 'ARL17B', 'CPVL', 'DPYD', 'FCGBP', 'FKBP5', 'HLAA', 'JAZF1', 'KCNIP4', 'KCNMA1', 'MEF2A', 'QKI', 'RASGEF1C', 'RPS19', 'SLC11A1', 'SPP1']

============================================================
Cell type: Inh
============================================================
AD predictors: 108
Validation predictors: 17
Overlap: 2
Odds Ratio: 22.4075
P-value: 4.6051e-03
Overlap genes (2):
['ARL17B', 'USP9Y']

============================================================
Cell type: Oli
============================================================
AD predictors: 621
Validation predictors: 9
Overlap: 3
Odds Ratio: 14.0008
P-value: 2.9695e-03
Overlap genes (3):
['PPP2R2B', 'QDPR', 'STMN4']

============================================================
Cell type: Ex
============================================================
AD predictors: 162
Validation predictors: 4
Overlap: 0
Odds Ratio: 0.0000
P-value: 1.0000e+00
Overlap genes (0):
[]

============================================================
Cell type: Opc
============================================================
AD predictors: 835
Validation predictors: 13
Overlap: 0
Odds Ratio: 0.0000
P-value: 1.0000e+00
Overlap genes (0):
[]

In [10]:
import os
import pandas as pd
import numpy as np

# -----------------------------
# Model directory mappings
# -----------------------------
rosmap_base_dirs = {
    "Combined": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_both_new/",
    "Genes": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new/",
    "Demo + APOE": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_demo_new/",
    "APOE + Genes": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_new/",
}

seaad_base_dirs = {
    "demographics_only": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_demographics_seaad/",
    "genes_demographics": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_demographics_seaad/",
    "genes_apoe": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_seaad/",
    "genes_only": "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad/",
}

# -----------------------------
# Match corresponding models
# -----------------------------
model_pairs = [
    ("Combined", "genes_demographics"),
    ("Genes", "genes_only"),
    ("Demo + APOE", "demographics_only"),
    ("APOE + Genes", "genes_apoe"),
]

# -----------------------------
# Cell type name mappings
# -----------------------------
cell_type_pairs = [
    ("Ast", "Ast"),
    ("Mic", "Mic"),
    ("In", "Inh"),
    ("Oli", "Oli"),
    ("Ex", "Ex"),
    ("Opc", "Opc"),
]

n_splits = 5

def get_mean_auc(base_dir, cell_type):
    aucs = []

    for split in range(1, n_splits + 1):
        output_path = os.path.join(base_dir, cell_type, f"split_{split}", "output_csv.csv")

        if not os.path.exists(output_path):
            continue

        try:
            df = pd.read_csv(output_path)
            if len(df) == 0:
                continue
            if "test_roc_auc" not in df.columns:
                continue

            auc_value = df.iloc[0]["test_roc_auc"]
            if pd.notnull(auc_value):
                aucs.append(float(auc_value))

        except Exception as e:
            print(f"Error reading {output_path}: {e}")

    if len(aucs) == 0:
        return np.nan, 0

    return np.mean(aucs), len(aucs)

# -----------------------------
# Print side-by-side results
# -----------------------------
print("=" * 120)
print("Mean test ROC AUC across splits: ROSMAP vs SEAAD")
print("=" * 120)

for rosmap_cell_type, seaad_cell_type in cell_type_pairs:
    print(f"\nCELL TYPE: ROSMAP={rosmap_cell_type} | SEAAD={seaad_cell_type}")
    print("-" * 120)

    for rosmap_model, seaad_model in model_pairs:
        rosmap_mean, rosmap_n = get_mean_auc(rosmap_base_dirs[rosmap_model], rosmap_cell_type)
        seaad_mean, seaad_n = get_mean_auc(seaad_base_dirs[seaad_model], seaad_cell_type)

        rosmap_str = f"{rosmap_mean:.4f} (n={rosmap_n})" if pd.notnull(rosmap_mean) else f"MISSING (n={rosmap_n})"
        seaad_str = f"{seaad_mean:.4f} (n={seaad_n})" if pd.notnull(seaad_mean) else f"MISSING (n={seaad_n})"

        print(f"{rosmap_model:<15} | ROSMAP: {rosmap_str:<18} | SEAAD: {seaad_str}")

print("\nDone.")

Mean test ROC AUC across splits: ROSMAP vs SEAAD

CELL TYPE: ROSMAP=Ast | SEAAD=Ast
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.7719 (n=5)       | SEAAD: 0.6220 (n=5)
Genes           | ROSMAP: 0.5678 (n=5)       | SEAAD: 0.5079 (n=5)
Demo + APOE     | ROSMAP: 0.6140 (n=5)       | SEAAD: 0.5004 (n=5)
APOE + Genes    | ROSMAP: 0.6230 (n=5)       | SEAAD: 0.5214 (n=5)

CELL TYPE: ROSMAP=Mic | SEAAD=Mic
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.5563 (n=5)       | SEAAD: 0.6352 (n=5)
Genes           | ROSMAP: 0.6531 (n=5)       | SEAAD: 0.5661 (n=5)
Demo + APOE     | ROSMAP: 0.3823 (n=5)       | SEAAD: 0.4934 (n=5)
APOE + Genes    | ROSMAP: 0.6507 (n=5)       | SEAAD: 0.6069 (n=5)

CELL TYPE: ROSMAP=In | SEAAD=Inh
---------------------------------------------------------------------

In [ ]:
========================================================================================================================
Mean test ROC AUC across splits: ROSMAP vs SEAAD
========================================================================================================================

CELL TYPE: ROSMAP=Ast | SEAAD=Ast
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.7719 (n=5)       | SEAAD: 0.6220 (n=5)
Genes           | ROSMAP: 0.5678 (n=5)       | SEAAD: 0.5079 (n=5)
Demo + APOE     | ROSMAP: 0.6140 (n=5)       | SEAAD: 0.5004 (n=5)
APOE + Genes    | ROSMAP: 0.6230 (n=5)       | SEAAD: 0.5214 (n=5)

CELL TYPE: ROSMAP=Mic | SEAAD=Mic
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.5563 (n=5)       | SEAAD: 0.6352 (n=5)
Genes           | ROSMAP: 0.6531 (n=5)       | SEAAD: 0.5661 (n=5)
Demo + APOE     | ROSMAP: 0.3823 (n=5)       | SEAAD: 0.4934 (n=5)
APOE + Genes    | ROSMAP: 0.6507 (n=5)       | SEAAD: 0.6069 (n=5)

CELL TYPE: ROSMAP=In | SEAAD=Inh
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.4490 (n=5)       | SEAAD: 0.6239 (n=5)
Genes           | ROSMAP: 0.6275 (n=5)       | SEAAD: 0.5069 (n=5)
Demo + APOE     | ROSMAP: 0.4658 (n=5)       | SEAAD: 0.5596 (n=5)
APOE + Genes    | ROSMAP: 0.6949 (n=5)       | SEAAD: 0.5430 (n=5)

CELL TYPE: ROSMAP=Oli | SEAAD=Oli
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.6575 (n=5)       | SEAAD: 0.5983 (n=5)
Genes           | ROSMAP: 0.6520 (n=5)       | SEAAD: 0.5731 (n=5)
Demo + APOE     | ROSMAP: 0.6741 (n=5)       | SEAAD: 0.5483 (n=5)
APOE + Genes    | ROSMAP: 0.6806 (n=5)       | SEAAD: 0.5090 (n=5)

CELL TYPE: ROSMAP=Ex | SEAAD=Ex
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.4970 (n=5)       | SEAAD: 0.6182 (n=5)
Genes           | ROSMAP: 0.5298 (n=5)       | SEAAD: 0.5487 (n=5)
Demo + APOE     | ROSMAP: 0.6375 (n=5)       | SEAAD: 0.5108 (n=5)
APOE + Genes    | ROSMAP: 0.6205 (n=5)       | SEAAD: 0.5362 (n=5)

CELL TYPE: ROSMAP=Opc | SEAAD=Opc
------------------------------------------------------------------------------------------------------------------------
Combined        | ROSMAP: 0.5262 (n=5)       | SEAAD: 0.7040 (n=5)
Genes           | ROSMAP: 0.5412 (n=5)       | SEAAD: 0.5887 (n=5)
Demo + APOE     | ROSMAP: 0.7732 (n=5)       | SEAAD: 0.6778 (n=5)
APOE + Genes    | ROSMAP: 0.6552 (n=5)       | SEAAD: 0.6759 (n=5)

Done.